In [6]:
import pandas as pd

In [7]:
df = pd.read_csv("IMDB Dataset.csv")

In [8]:
df.shape

(50000, 2)

In [9]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [10]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [11]:
df.duplicated().sum()

np.int64(418)

In [12]:
df.drop_duplicates(inplace =True)

In [13]:
df.shape

(49582, 2)

## Pre - Processing

### 1 Converting to lowecase

In [14]:
df["review"] = df["review"].str.lower() # 


### 2. Removing the URLs 

In [15]:
import re

# sample_text = "abc is the word ,abc" # abc => xyz

# new_text = re.sub("abc","xyz",sample_text)


In [16]:
def remove_urls(text):
    text = re.sub(r"http\S+" ,"",text) # (pattern , repl, string) \S = nonwhite space charaacter , \s = white space cahracter 
    return text

df["review"] = df["review"].apply(remove_urls)

### 3.Removing punctuations

In [17]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]","",text)
    return text 

df["review"] = df["review"].apply(remove_punctuation)

In [18]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. Removing HTML

In [19]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text)
    return text 

df["review"] = df["review"].apply(remove_html)

### 5. Removing the stopwords

In [20]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


In [22]:
# sample_text = "I like coding in python"
# tokens = word_tokenize(sample_text)

In [24]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    
    return text    

df["review"] = df["review"].apply(remove_stopwords)

In [26]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. stemming

In [27]:
# running  = run 
#  played = play
#  porterstemming

from nltk.stem import PorterStemmer


In [32]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)      #join() list ke words ko spaces ke saath jod deta hai.

df["review"] = df["review"].apply(stemming)

In [33]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,1
1,wder ltle producti br br film techniqu unssum ...,1
2,thought th wder wy spend tme o hot summer week...,1
3,bsclli re fmli lttle boy jke thk re zomb close...,0
4,petter mtte love time mey vulli stunng film wt...,1


### 7. Encoding

In [30]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])


In [31]:

y = df["sentiment"]

In [ ]:
y

### 8. vectorization

In [ ]:
df.head()


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)
X = tf.fit_transform(df["review"])
print(type(X))
# tf.get_feature_names_out()
# X.toarray()
print(X)

### Dataset and DataLoader

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test , y_train, y_test = train_test_split(
    X , y, test_size = 0.2, random_state = 42
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
import torch
from torch.utils.data import TensorDataset , DataLoader

In [ ]:
X_train

In [ ]:
import numpy as np

X_train = X_train.toarray()
X_test = X_test.toarray()
print(X_train.shape)
print(X_train.size)

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set ,shuffle = True, batch_size=64)
test_loader = DataLoader(test_set ,shuffle = True, batch_size=64)

## Build our RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim


In [ ]:


class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN
        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # shape: (num_layers, batch_size, hidden_size)
        h0 = torch.zeros(
            self.num_layers,
            x.size(0),
            self.hidden_size
        )

        out, _ = self.rnn(x, h0)
        # 1. Hidden state of all the timestep === # out shape: (batch, seq_len, hidden_size)
        # 2.final hidden state of last timestep 
        
        # Take output from last timestep
        out = self.fc(out[:, -1, :])

        return out

In [ ]:
input_size = X_train.shape[1]
model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

### Training the RNN

In [ ]:
# unsqueeze  , squeeze

epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size ,) => probability
        loss = criterion(outputs,yb) # compute loss
        loss.backward()
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        

In [ ]:
# Evaluate 

model.eval()
with torch.no_grad():

    correct_vals = 0
    tot_vals = 0

    for Xb , yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_vals += yb.size(0)
        correct_vals+= (predicted ==yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")